In [0]:
%sql
CREATE WIDGET TEXT prm_start_year_month DEFAULT "201001";
CREATE WIDGET TEXT prm_end_year_month DEFAULT "202602";

In [0]:
from pyspark.sql import functions as F

BASE_RULES = {
    "missing_pickup_datetime": F.col("pickup_datetime").isNull(),
    "missing_dropoff_datetime": F.col("dropoff_datetime").isNull(),
    "dropoff_before_pickup": (F.col("pickup_datetime").isNotNull())
    & (F.col("dropoff_datetime").isNotNull())
    & (F.col("dropoff_datetime") < F.col("pickup_datetime")),
    "passenger_count_negative": F.col("passenger_count") < 0,
    "trip_distance_too_small": F.col("trip_distance") < 0.1,
    "total_amount_negative": F.col("total_amount") < 0,
    "invalid_YYYYMM": F.col("YYYYMM").isNotNull()
    & ((F.col("YYYYMM") < 190001) | (F.col("YYYYMM") > 300012)),
    "missing_fare_amount": F.col("fare_amount").isNull(),
    "invalid_fare_amount": F.col("fare_amount").isNotNull() & (F.col("fare_amount") < 2.5),
    "invalid_duration_min": (F.col("pickup_datetime").isNotNull())
    & (F.col("dropoff_datetime").isNotNull())
    & ((F.col("duration_min") < 2.0) | (F.col("duration_min") > 300)), 
    "impossible_speed": (F.col("trip_distance") / (F.col("duration_min") / 60) > 100 )
    
}

#### Calculate the duration min column

In [0]:
# get parameter for start and end year month
start_ym = int(dbutils.widgets.get("prm_start_year_month"))
end_ym = int(dbutils.widgets.get("prm_end_year_month"))

In [0]:
df_silver_nyc_yellow = (
    spark.table("nyc.process_bronze.brz_yellow_nyc_taxi")
    .filter(F.col("YYYYMM").between(start_ym, end_ym))
)

In [0]:
# Calculate duration in minutes and mark timestamp for data lineage
df_silver_nyc_yellow_enriched = (
    df_silver_nyc_yellow.withColumn(
    "duration_min", 
    (F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")) / 60 )
    .withColumnRenamed("_load_timestamp", "brz_load_timestamp") 
    .withColumn("slv_load_timestamp", F.current_timestamp())
)


In [0]:
# Set clean definition
bad_data_condition = F.lit(False) 
for rule_name, condition in BASE_RULES.items(): 
    bad_data_condition = bad_data_condition | condition 


df_silver_nyc_yellow_clean = df_silver_nyc_yellow_enriched.filter(~bad_data_condition) 


In [0]:
# Append new data into delta table 
(
    df_silver_nyc_yellow_clean.write 
        .mode("overwrite") 
        .format("")
)